# 04 · Retrieve — 02 Multi-Source Fan-Out (the full path)

**4 of 6 sources are free and keyless (PubMed, OpenAlex, ClinicalTrials.gov, and a local vector stand-in); Semantic Scholar is keyless but rate-limited (real 429 below); `TAVILY_API_KEY` unlocks the web-search leg.**

Implements a fan-out across six source clients queried in parallel: a
vector index, PubMed, Semantic Scholar, Tavily web search,
ClinicalTrials.gov, and OpenAlex.

This notebook makes real network calls to public APIs. No credential is ever
read from a `.env` file in this repo before being used here --
`nbio.show_environment()` below prints only which keys are *loaded*, never a
value.

**The one thing this notebook exists to demonstrate:** each source is wrapped
so a failure never raises past the fan-out. That is correct, defensive
behavior -- and it is also exactly how a *total* retrieval failure looks
identical to a healthy run, because six empty lists sum to zero results
either way, whether every source is silent or every source is broken. So
every source below prints its own document count, and a source that fails
prints its exception instead of vanishing into an empty list. That guardrail
(`_run_source`) is built and proven first, below, before any of the six real
source clients exist.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `_run_source` | The guardrail: runs one source call, prints its count on success or its real exception on failure -- never a silent empty list. | `await _run_source("pubmed", search_pubmed, QUERY, 8)` |
| `search_pubmed` | PubMed E-utilities search -- free, keyless. | `search_pubmed(QUERY, max_results=8)` |
| `search_semantic_scholar` | Semantic Scholar search -- keyless, but rate-limited (real 429 without a key). | `search_semantic_scholar(QUERY, max_results=8)` |
| `search_openalex` | OpenAlex works search, with abstract-inverted-index decoding -- free, keyless. | `search_openalex(QUERY, max_results=8)` |
| `search_web_tavily` | Tavily web search restricted to a medical-domain allowlist -- needs `TAVILY_API_KEY`. | `search_web_tavily(QUERY)` |
| `search_clinical_trials` | ClinicalTrials.gov v2 API -- free, keyless. | `search_clinical_trials(QUERY)` |
| `search_vectors_local` | Local hash-embedding stand-in for a real Pinecone/S3 Vectors index. | `search_vectors_local(QUERY, top_k=8)` |
| `search_bm25_local` | Bonus: sparse (BM25) ranking over the same 4 synthetic documents, no embeddings. | `search_bm25_local(QUERY, top_k=8)` |
| `search_papers_fanout` | Runs all six sources in parallel via the guardrail wrapper and gathers the results. | `await search_papers_fanout(QUERY, top_k=8)` |


In [ ]:
# Locate the repo root before `import nbio` can work at all -- see the same
# shim in 01-single-index-retrieval.ipynb for why this is needed.
import sys
from pathlib import Path

_p = Path.cwd().resolve()
for _ in range(6):
    if (_p / "nbio.py").is_file():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")

import nbio

repo_root = nbio.bootstrap()
nbio.show_environment()

## A deliberate design choice: don't swallow exceptions in the client

A common pattern for source clients (`pubmed.py`, `semantic_scholar.py`,
`openalex.py`, `web_search.py`, `clinical_trials.py`) is to wrap each network
call in a `try/except Exception: return []` *inside the client itself* -- so
by the time a fan-out-level `_safe()` wrapper gets a chance to catch
anything, the real exception is already gone, logged at `debug`/`warning`
level at best and easy to miss.

The source functions below do **not** swallow their own exceptions. A
network or HTTP error propagates out of the source function and is caught
exactly once, at the fan-out layer, by the `_run_source` wrapper defined
further down -- which prints it. That is the fix this notebook is built to
demonstrate.

## Step 1 — the guardrail: `_run_source`

This is the fix the previous cell describes, built first, before any of the
six real source clients exist. It runs one source call in a thread (the
source functions are all blocking `urllib` calls) and either prints a
document count or prints the real exception -- never converts both outcomes
into an indistinguishable empty list.

In [ ]:
import asyncio


async def _run_source(name: str, func, *args) -> list[dict]:
    loop = asyncio.get_running_loop()
    try:
        docs = await loop.run_in_executor(None, lambda: func(*args))
        docs = docs or []
        print(f"  {name:<18}: {len(docs):3d} documents")
        return docs
    except Exception as exc:  # noqa: BLE001 -- this is the one place we deliberately catch broadly, to print
        print(f"  {name:<18}: FAILED -- {type(exc).__name__}: {exc}")
        return []

## Step 2 — prove the guardrail actually catches a failure

Before building a single real source client, show `_run_source` doing its
one job on two synthetic calls: one that succeeds, one that raises. This is
the guardrail this notebook is built to demonstrate, working, before any
capability is built on top of it.

In [ ]:
def _demo_ok_source():
    return [{"title": "synthetic result"}]


def _demo_failing_source():
    raise RuntimeError("synthetic failure -- this is what a broken source looks like")


_ = await _run_source("demo_ok", _demo_ok_source)
_ = await _run_source("demo_failing", _demo_failing_source)

## Step 3 — the real query used throughout this notebook

One of the twenty published benchmark questions this repo ships for stage
`06-bench` (`clinical_queries.py`, id `W01`) -- a real question, not patient
data.

In [ ]:
QUERY = "Negative pressure wound therapy for diabetic foot ulcers: outcomes evidence"

## Step 4 — source: PubMed — E-utilities, no key required

Uses PubMed's E-utilities: esearch to get PMIDs, efetch to get the article
XML, parsed into paper dicts.

In [ ]:
import json
import re
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET

PUBMED_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"


def _urlopen(url: str, timeout: int = 20) -> bytes:
    req = urllib.request.Request(url, headers={"User-Agent": "anacodic-cookbook/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read()


def _normalize_year(raw):
    if not raw:
        return None
    m = re.search(r"(\d{4})", str(raw))
    return int(m.group(1)) if m else None


def _extract_doi(article) -> str:
    for aid in article.findall(".//ArticleId"):
        if (aid.get("IdType") or "").lower() == "doi":
            return (aid.text or "").strip()
    return ""


def _extract_pmc_id(article) -> str:
    for aid in article.findall(".//ArticleId"):
        if (aid.get("IdType") or "").lower() == "pmc":
            return (aid.text or "").strip()
    return ""


def _extract_abstract(article) -> str:
    parts = []
    for abt in article.findall(".//Abstract/AbstractText"):
        txt = (abt.text or "").strip()
        label = (abt.get("Label") or "").strip()
        if txt:
            parts.append(f"{label}: {txt}" if label else txt)
    return "\n".join(parts).strip()


def _parse_pubmed_xml(xml_text: str) -> list[dict]:
    papers = []
    root = ET.fromstring(xml_text)
    for article in root.findall(".//PubmedArticle"):
        medline = article.find(".//MedlineCitation")
        if medline is None:
            continue
        art = medline.find(".//Article")
        if art is None:
            continue
        title = (art.findtext(".//ArticleTitle") or "").strip()
        journal = (art.findtext(".//Journal/Title") or "").strip()
        year_raw = (
            art.findtext(".//Journal/JournalIssue/PubDate/Year")
            or art.findtext(".//Journal/JournalIssue/PubDate/MedlineDate")
            or ""
        )
        abstract = _extract_abstract(article)
        doi = _extract_doi(article)
        pmcid = _extract_pmc_id(article)
        pmc_url = f"https://pmc.ncbi.nlm.nih.gov/articles/{pmcid}/" if pmcid else ""
        if not title and not abstract:
            continue
        papers.append(
            {
                "title": title,
                "text": abstract,
                "pmcid": pmcid,
                "pmc_url": pmc_url,
                "doi": doi,
                "year": _normalize_year(year_raw),
                "journal": journal,
                "source": "pubmed",
                "score": 0.5,
            }
        )
    return papers


def search_pubmed(query: str, max_results: int = 10) -> list[dict]:
    q = (query or "").strip()
    if not q:
        return []
    params = {"db": "pubmed", "term": q, "retmax": max_results, "sort": "relevance", "retmode": "json"}
    url = f"{PUBMED_BASE}/esearch.fcgi?{urllib.parse.urlencode(params)}"
    data = json.loads(_urlopen(url).decode("utf-8"))
    pmids = [p for p in data.get("esearchresult", {}).get("idlist", []) if isinstance(p, str) and p.strip()]
    if not pmids:
        return []
    detail_params = {"db": "pubmed", "id": ",".join(pmids), "retmode": "xml", "rettype": "abstract"}
    xml_url = f"{PUBMED_BASE}/efetch.fcgi?{urllib.parse.urlencode(detail_params)}"
    xml_text = _urlopen(xml_url).decode("utf-8", errors="ignore")
    return _parse_pubmed_xml(xml_text)

## Step 5 — run PubMed alone, through the guardrail wrapper

In [ ]:
_ = await _run_source("pubmed", search_pubmed, QUERY, 8)

## Step 6 — source: Semantic Scholar — free API, keyless calls are rate-limited

Uses the Semantic Scholar search endpoint only -- citation enrichment and
citation-graph traversal are a larger surface that isn't needed for the
fan-out itself.

`SEMANTIC_SCHOLAR_API_KEY` is sent as `x-api-key` if present; if it's absent,
this is the exact call that returns HTTP 429 in practice (known defect 2, see
this stage's `README.md`). It also does **not** swallow its own exception --
see "A deliberate design choice" above -- so a 429 propagates out to
`_run_source`, which is exactly where it should be caught.

In [ ]:
import os

S2_BASE = "https://api.semanticscholar.org/graph/v1"
S2_SEARCH_FIELDS = "title,abstract,year,citationCount,venue,externalIds,isOpenAccess,url"


def _s2_headers() -> dict:
    key = os.environ.get("SEMANTIC_SCHOLAR_API_KEY", "").strip()
    h = {"User-Agent": "anacodic-cookbook/1.0"}
    if key:
        h["x-api-key"] = key
    return h


def _extract_doi_from_external_ids(external_ids) -> str:
    if not isinstance(external_ids, dict):
        return ""
    return str(external_ids.get("DOI") or external_ids.get("doi") or "").strip()


def _extract_venue_name(payload: dict) -> str:
    if isinstance(payload.get("venue"), dict):
        return str(payload["venue"].get("name") or "").strip()
    return str(payload.get("venue") or "").strip()


def _parse_s2_paper_payload(p: dict) -> dict:
    external_ids = p.get("externalIds") or {}
    doi = _extract_doi_from_external_ids(external_ids) or p.get("doi") or ""
    return {
        "title": str(p.get("title") or "").strip(),
        "year": p.get("year"),
        "journal": _extract_venue_name(p),
        "text": str(p.get("abstract") or "").strip(),
        "doi": str(doi).strip(),
        "pmcid": "",
        "pmc_url": "",
        "citation_count": p.get("citationCount"),
        "source": "s2",
        "score": 0.5,
    }


def search_semantic_scholar(query: str, max_results: int = 10) -> list[dict]:
    q = (query or "").strip()
    if not q:
        return []
    params = {"query": q, "limit": max(1, min(20, int(max_results))), "fields": S2_SEARCH_FIELDS}
    url = f"{S2_BASE}/paper/search?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers=_s2_headers())
    # NOTE: no try/except here -- see "A deliberate departure" above. A 429
    # raises urllib.error.HTTPError, which propagates to _run_source below.
    with urllib.request.urlopen(req, timeout=20) as resp:
        data = json.loads(resp.read().decode("utf-8", errors="ignore"))
    payloads = data.get("data") if isinstance(data, dict) else None
    if not payloads:
        return []
    return [_parse_s2_paper_payload(p) for p in payloads if isinstance(p, dict)]

## Step 7 — run Semantic Scholar alone — the real 429 in practice

Without `SEMANTIC_SCHOLAR_API_KEY` set, this line is expected to print
`FAILED -- HTTPError: HTTP Error 429: Too Many Requests`, not a silent `0`.

In [ ]:
_ = await _run_source("semantic_scholar", search_semantic_scholar, QUERY, 8)

## Step 8 — source: OpenAlex — free, keyless

Includes an `abstract_inverted_index` decoder (OpenAlex stores abstracts as
a word→positions index rather than plain text, for copyright reasons).

In [ ]:
OPENALEX_BASE = "https://api.openalex.org"


def _best_abstract(iab) -> str:
    if not isinstance(iab, dict) or not iab:
        return ""
    pos_to_word: dict[int, str] = {}
    for word, positions in iab.items():
        if not isinstance(word, str) or not isinstance(positions, list):
            continue
        for p in positions:
            if isinstance(p, int):
                pos_to_word[p] = word
    if not pos_to_word:
        return ""
    return " ".join(pos_to_word[i] for i in sorted(pos_to_word)).strip()


def search_openalex(query: str, max_results: int = 10) -> list[dict]:
    q = (query or "").strip()
    if not q:
        return []
    params = {
        "search": q,
        "per-page": max(1, min(25, int(max_results))),
        "select": ",".join(
            ["id", "display_name", "publication_year", "doi", "primary_location", "abstract_inverted_index"]
        ),
    }
    url = f"{OPENALEX_BASE}/works?{urllib.parse.urlencode(params)}"
    req = urllib.request.Request(url, headers={"User-Agent": "anacodic-cookbook/1.0"})
    with urllib.request.urlopen(req, timeout=25) as resp:
        data = json.loads(resp.read().decode("utf-8", errors="ignore"))
    results = data.get("results") if isinstance(data, dict) else None
    if not isinstance(results, list):
        return []
    out = []
    for r in results[:max_results]:
        title = str(r.get("display_name") or "").strip()
        doi = str(r.get("doi") or "").strip().replace("https://doi.org/", "")
        pl = r.get("primary_location") or {}
        journal = str((pl.get("source") or {}).get("display_name") or "").strip()
        abstract = _best_abstract(r.get("abstract_inverted_index"))
        if not abstract and not title:
            continue
        out.append(
            {
                "title": title,
                "text": abstract,
                "doi": doi,
                "pmcid": "",
                "pmc_url": "",
                "year": r.get("publication_year"),
                "journal": journal,
                "source": "openalex",
                "score": 0.5,
            }
        )
    return out

## Step 9 — run OpenAlex alone, through the guardrail wrapper

In [ ]:
_ = await _run_source("openalex", search_openalex, QUERY, 8)

## Step 10 — source: Tavily web search — requires `TAVILY_API_KEY`

Restricted to a medical-domain allowlist by default. Rather than logging a
warning and returning `[]` when the key is missing, this raises so the
fan-out's per-source line says exactly why the count is zero.

In [ ]:
TAVILY_URL = "https://api.tavily.com/search"
DEFAULT_MEDICAL_DOMAINS = ["nccn.org", "fda.gov", "who.int", "nejm.org", "jamanetwork.com", "cochranelibrary.com"]


def search_web_tavily(query: str, max_results: int = 8) -> list[dict]:
    key = os.environ.get("TAVILY_API_KEY", "").strip()
    if not key:
        raise RuntimeError("TAVILY_API_KEY not set -- web search source is disabled")
    payload = {
        "api_key": key,
        "query": query.strip(),
        "search_depth": "advanced",
        "include_answer": False,
        "max_results": max_results,
        "include_domains": DEFAULT_MEDICAL_DOMAINS,
    }
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        TAVILY_URL, data=data, headers={"Content-Type": "application/json"}, method="POST"
    )
    with urllib.request.urlopen(req, timeout=25) as resp:
        result = json.loads(resp.read().decode("utf-8", errors="ignore"))
    out = []
    for r in result.get("results") or []:
        title = str(r.get("title") or "").strip()
        content = str(r.get("content") or "").strip()
        if not content and not title:
            continue
        out.append(
            {
                "title": title,
                "text": content,
                "doi": "",
                "pmcid": "",
                "pmc_url": r.get("url", ""),
                "year": None,
                "journal": "",
                "source": "web",
                "score": 0.5,
            }
        )
    return out

## Step 11 — run Tavily alone — fails cleanly without a key

Without `TAVILY_API_KEY` set, this is expected to print
`FAILED -- RuntimeError: TAVILY_API_KEY not set -- web search source is
disabled` -- the exact guardrail behavior this notebook is about, on a
second, independent source.

In [ ]:
_ = await _run_source("web (tavily)", search_web_tavily, QUERY)

## Step 12 — source: ClinicalTrials.gov — free, keyless

Queries the public ClinicalTrials.gov API directly, no key required.

In [ ]:
CTGOV_BASE = "https://clinicaltrials.gov/api/v2/studies"


def search_clinical_trials(query: str, max_studies: int = 10) -> list[dict]:
    q = (query or "").strip()
    if not q:
        return []
    params = {"query.term": q, "pageSize": min(10, max_studies), "format": "json"}
    url = f"{CTGOV_BASE}?{urllib.parse.urlencode(params)}"
    req = urllib.request.Request(url, headers={"User-Agent": "anacodic-cookbook/1.0"})
    with urllib.request.urlopen(req, timeout=25) as resp:
        data = json.loads(resp.read().decode("utf-8", errors="ignore"))
    studies = data.get("studies") if isinstance(data, dict) else None
    if not isinstance(studies, list):
        return []
    out = []
    for study in studies[:max_studies]:
        proto = study.get("protocolSection") or {}
        ident = proto.get("identificationModule") or {}
        desc = proto.get("descriptionModule") or {}
        nct = str(ident.get("nctId") or "").strip()
        title = str(ident.get("briefTitle") or "").strip()
        brief = str(desc.get("briefSummary") or "").strip()
        if not title and not brief:
            continue
        out.append(
            {
                "title": title or nct,
                "text": brief or title,
                "doi": "",
                "pmcid": "",
                "pmc_url": f"https://clinicaltrials.gov/study/{nct}" if nct else "",
                "year": None,
                "journal": "ClinicalTrials.gov",
                "source": "clinicaltrials",
                "score": 0.5,
                "nct_id": nct,
            }
        )
    return out

## Step 13 — run ClinicalTrials.gov alone, through the guardrail wrapper

In [ ]:
_ = await _run_source("clinical_trials", search_clinical_trials, QUERY)

## Step 14 — source: local vector index (stand-in for Pinecone/S3 Vectors)

In production, `vector_search.py::search_vectors` routes to a real Pinecone or
S3 Vectors index built in stage `03-embed` over a real corpus. This repo
doesn't ship a populated index, so this leg is a tiny, openly-invented
4-document stand-in using the same hash-embedding fallback as notebook `01` --
just enough to exercise the "vectors" position in the fan-out, not a claim
about real semantic search quality.

In [ ]:
import hashlib
import math


def embed_text(text: str, dim: int = 64) -> list[float]:
    vec = [0.0] * dim
    for tok in (text or "").lower().split():
        h = int(hashlib.sha256(tok.encode("utf-8")).hexdigest(), 16)
        vec[h % dim] += 1.0
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]


def _cosine(a: list[float], b: list[float]) -> float:
    if not a or not b or len(a) != len(b):
        return 0.0
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a)) or 1.0
    nb = math.sqrt(sum(x * x for x in b)) or 1.0
    return dot / (na * nb)


_LOCAL_INDEX = [
    {
        "title": "Negative pressure wound therapy in chronic lower-limb ulcers (synthetic entry)",
        "text": "Negative pressure wound therapy is widely used as an adjunct in chronic "
        "wound management, including diabetic foot ulcers, with reported effects on "
        "granulation tissue formation and exudate control.",
        "doi": "",
        "pmcid": "",
        "pmc_url": "",
        "year": 2021,
        "journal": "Wound Repair and Regeneration",
        "source": "vectors",
    },
    {
        "title": "Standard dressing protocols for diabetic foot ulcers (synthetic entry)",
        "text": "Standard moist wound dressings remain the comparator arm in most diabetic "
        "foot ulcer trials, with healing rates used as the primary outcome measure.",
        "doi": "",
        "pmcid": "",
        "pmc_url": "",
        "year": 2018,
        "journal": "Wound Repair and Regeneration",
        "source": "vectors",
    },
    {
        "title": "Amputation risk reduction strategies in diabetic foot disease (synthetic entry)",
        "text": "Multidisciplinary foot-care teams and early offloading are associated with "
        "reduced major amputation rates in patients with diabetic foot ulceration.",
        "doi": "",
        "pmcid": "",
        "pmc_url": "",
        "year": 2020,
        "journal": "Diabetes Care",
        "source": "vectors",
    },
    {
        "title": "Gutenberg-era printing techniques (synthetic entry, off-topic control)",
        "text": "Movable type printing spread across Europe rapidly after the mid-15th "
        "century, changing how texts were reproduced and distributed.",
        "doi": "",
        "pmcid": "",
        "pmc_url": "",
        "year": 1970,
        "journal": "History of Technology",
        "source": "vectors",
    },
]
for _d in _LOCAL_INDEX:
    _d["embedding"] = embed_text(_d["text"])


def search_vectors_local(query: str, top_k: int = 10) -> list[dict]:
    qvec = embed_text(query)
    scored = []
    for d in _LOCAL_INDEX:
        rec = {k: v for k, v in d.items() if k != "embedding"}
        rec["score"] = _cosine(qvec, d["embedding"])
        scored.append(rec)
    scored.sort(key=lambda x: -x["score"])
    return scored[:top_k]

## Step 15 — run the local vector index alone, through the guardrail wrapper

In [ ]:
_ = await _run_source("vectors", search_vectors_local, QUERY, 8)

## Step 16 — bonus: the sparse-retrieval leg (BM25)

In production, sparse retrieval loads a *fitted* `pinecone-text`
`BM25Encoder` pickle to blend sparse (BM25) and dense vectors for Pinecone
hybrid search -- that pickle is a build artifact, not something worth
vendoring here. `rank-bm25` (a small, dependency-free BM25 implementation)
stands in to show the same idea -- ranking the same four synthetic
documents by term overlap alone, no embeddings at all -- so you can compare
it with the dense result above. This is a bonus leg, not one of the six
sources the fan-out below actually gathers.

In [ ]:
from rank_bm25 import BM25Okapi

_corpus_tokens = [d["text"].lower().split() for d in _LOCAL_INDEX]
_bm25 = BM25Okapi(_corpus_tokens)


def search_bm25_local(query: str, top_k: int = 10) -> list[dict]:
    scores = _bm25.get_scores(query.lower().split())
    ranked = sorted(zip(_LOCAL_INDEX, scores), key=lambda x: -x[1])
    return [{**{k: v for k, v in d.items() if k != "embedding"}, "score": float(s)} for d, s in ranked[:top_k]]

## Step 17 — run BM25 alone, and compare with the dense result above

In [ ]:
_ = await _run_source("bm25_local (bonus)", search_bm25_local, QUERY, 8)

## Step 18 — the full fan-out: all six sources in parallel

Each source runs in a thread via `asyncio.get_running_loop().run_in_executor`,
gathered in parallel with `asyncio.gather`, using the exact `_run_source`
guardrail proven in Step 2 above and exercised individually on every real
source since -- nothing new is introduced here except running them
together.

In [ ]:
async def search_papers_fanout(query: str, top_k: int = 10) -> dict[str, list[dict]]:
    print(f"fan-out query: {query!r}\n")
    names_and_calls = [
        ("vectors", search_vectors_local, (query, top_k)),
        ("pubmed", search_pubmed, (query, top_k)),
        ("semantic_scholar", search_semantic_scholar, (query, top_k)),
        ("web (tavily)", search_web_tavily, (query,)),
        ("clinical_trials", search_clinical_trials, (query,)),
        ("openalex", search_openalex, (query, top_k)),
    ]
    results = await asyncio.gather(*[_run_source(name, func, *args) for name, func, args in names_and_calls])
    return dict(zip((n for n, _, _ in names_and_calls), results))

## Step 19 — run it and look at the merged totals

In [ ]:
by_source = await search_papers_fanout(QUERY, top_k=8)

naive_total = sum(len(v) for v in by_source.values())
print(f"\nnaive total across sources (pre-dedup): {naive_total}")

## Why the per-source count is the important number

If every one of the six sources above happened to return zero documents --
a bad query, a network outage, six simultaneous API changes, whatever the
cause -- the fan-out's *output* would look exactly like this run's healthy
result, minus the per-source lines: an empty merged list. `naive_total` alone
cannot distinguish "nothing relevant exists" from "every source silently
failed." The per-source table above is what makes that distinction visible;
without it, a total retrieval failure and a successful run of an
unanswerable query render identically.

Semantic Scholar's line above is the concrete case: if `SEMANTIC_SCHOLAR_API_KEY`
isn't set, that source contributes `0` and its line names the exact reason
(`HTTPError: HTTP Error 429: Too Many Requests`) rather than blending into a
row of unremarkable zeros. Register a free key at
semanticscholar.org/product/api, set `SEMANTIC_SCHOLAR_API_KEY`, and rerun the
cell above -- the `semantic_scholar` line should turn into a real count.

Next: `03-dedupe-and-merge.ipynb` turns these six per-source lists into one
deduplicated candidate list.